# 🔥 Phase 2: Fine-tune Phi-3-mini with QLoRA
### PyTorch Debug Assistant

**What this notebook does:**
1. Installs dependencies
2. Loads data/processed/structured_train.jsonl from the repo
3. Fine-tunes Phi-3-mini to return structured debugging JSON
4. Pushes the adapter to HuggingFace Hub

**Runtime:** Set to GPU T4 x2 in Kaggle Settings

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q -U huggingface_hub
!pip install -q -U bitsandbytes
!pip install -q triton
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q trl==0.10.1
!pip install -q datasets==2.21.0
!pip install -q accelerate==0.34.2
!pip install -q wandb

## Step 2 — Authenticate with HuggingFace

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_TOKEN = user_secrets.get_secret("WANDB_API_KEY")

assert HF_TOKEN, "HF_TOKEN is missing. Enable it in Kaggle Add-ons → Secrets."
assert WANDB_TOKEN, "WANDB_API_KEY is missing. Enable it in Kaggle Add-ons → Secrets."

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_TOKEN

login(token=HF_TOKEN)
print("Hugging Face login ✅")

wandb.login(key=WANDB_TOKEN)
print("W&B login ✅")

In [ ]:
import os

if not os.path.exists("/kaggle/working/pytorch-debug-assistant"):
    !git clone https://github.com/usazehan/pytorch-debug-assistant.git

%cd /kaggle/working/pytorch-debug-assistant
!git pull

In [ ]:
!ls data/processed
!wc -l data/processed/structured_train.jsonl

## Step 3 — Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found. Enable GPU T4 x2 in Kaggle settings."

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

## Step 4 — Load and Inspect Dataset

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset('zehansunesara/pytorch-debug-assistant')
# print(dataset)
# print('\n--- Sample ---')
# print('INPUT:', dataset['train'][0]['input'][:300])
# print('\nOUTPUT:', dataset['train'][0]['output'][:300])
import json
from datasets import Dataset

structured_path = "data/processed/structured_train.jsonl"

rows = []
with open(structured_path, "r") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

dataset = Dataset.from_list(rows)
dataset

## Step 5 — Format into Chat Template

Phi-3-mini uses `<|user|>...<|end|>` format. We convert our instruction/input/output triplets into this format.

In [ ]:
import json

SYSTEM_PROMPT = """You are a PyTorch debugging assistant.

Given a PyTorch error, question, and code context, classify the issue and explain the fix.

Return ONLY a valid JSON object with exactly these fields:
{
  "category": "...",
  "root_cause": "...",
  "fix": "...",
  "fix_code": "..."
}

The category must be exactly one of:
- tensor_shape_mismatch
- cuda_oom
- device_mismatch
- dtype_mismatch
- autograd_error
- dataloader_error
- loss_issue
- environment_error
- optimizer_error
- training_loop_bug
- architecture_mismatch

Category decision rules:
- Use "cuda_oom" ONLY when the error explicitly says CUDA out of memory, out of memory, or the process was killed because of memory pressure.
- Use "environment_error" for install/import/version/driver/CUDA setup issues, including "no NVIDIA driver", "Torch not compiled with CUDA enabled", "torch.cuda.is_available() returns False", missing packages, or CUDA/cuDNN version mismatch.
- Use "loss_issue" for NaN loss, loss not decreasing, exploding gradients, parameters becoming NaN/zero during training, or unstable optimization behavior.
- Use "dtype_mismatch" for Float/Double/Half/Long/Byte tensor type mismatches.
- Use "device_mismatch" for CPU vs CUDA tensor placement problems.
- Use "tensor_shape_mismatch" for tensor size, dimension, broadcasting, matmul, or reshape/view problems.
- Use "architecture_mismatch" for layer/channel mismatches, state_dict loading problems, missing/unexpected keys, or incompatible model heads.
- Use "training_loop_bug" for device-side asserts, invalid target labels, incorrect loss usage, or mistakes in the train/eval loop.
- Use "optimizer_error" only for optimizer construction/state issues like empty parameter lists or invalid optimizer state.

Keep root_cause and fix to one concise sentence each.
Keep fix_code to one minimal code snippet or an empty string.
Do not include markdown.
Do not include explanations outside the JSON.
"""


def format_structured_example(example):
    inp = example["input"]
    out = example["output"]

    user_prompt = f"""Classify this PyTorch debugging issue.

Question title:
{inp.get("question_title", "")}

Error text:
{inp.get("error_text", "")}

Code context:
{inp.get("code_context", "")}

Question body:
{inp.get("question_body", "")}

Return JSON only.
"""

    assistant_json = json.dumps(
        {
            "category": out["category"],
            "root_cause": out["root_cause"],
            "fix": out["fix"],
            "fix_code": out["fix_code"],
        },
        ensure_ascii=False,
    )

    return {
        "text": (
            "<|system|>\n"
            f"{SYSTEM_PROMPT}<|end|>\n"
            "<|user|>\n"
            f"{user_prompt}<|end|>\n"
            "<|assistant|>\n"
            f"{assistant_json}<|end|>"
        )
    }


formatted_dataset = dataset.map(format_structured_example)

split = formatted_dataset.train_test_split(test_size=0.15, seed=42)

train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print("\n--- Formatted sample ---")
print(train_dataset[0]["text"][:1000])

## Step 6 — Load Phi-3-mini-4k-instruct in 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NormalFloat4 — best quality at 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # T4 uses fp16
    bnb_4bit_use_double_quant=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # required for SFTTrainer

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.config.use_cache = False  # disable KV cache during training

print(f'Model loaded ✅')
print(f'Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 7 — Configure LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (freezes base weights, casts norms to fp32)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity, more params
    lora_alpha=32,           # scaling factor (alpha/r = effective LR scale)
    target_modules=[         # which layers to add LoRA to
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # attention
        'gate_proj', 'up_proj', 'down_proj'       # MLP
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# Show trainable params — should be ~1-2% of total
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of total)')

## Step 8 — Training Configuration

In [32]:
SMOKE_TEST = True

In [34]:
from transformers import TrainingArguments

if SMOKE_TEST:
    training_args = TrainingArguments(
        output_dir="./phi3-structured-v3-smoke",
        run_name="phi3-structured-v3-smoke",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=1,
        max_steps=25,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=1,
        fp16=True,
        report_to="wandb",
        load_best_model_at_end=False,
    )
else:
    training_args = TrainingArguments(
        output_dir="./phi3-structured-v3",
        run_name="phi3-structured-v3",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=1,
        max_steps=75,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=2,
        fp16=True,
        report_to="wandb",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

## Step 9 — Train!

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print('Starting training...')
trainer.train()
print('Training complete ✅')

## Step 10 — Evaluate: Before vs After

In [ ]:
import torch

# Test prompt — a real PyTorch error
test_input = {
    "question_title": "Expected all tensors to be on the same device",
    "error_text": "RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!",
    "code_context": "loss = criterion(outputs, labels)",
    "question_body": "My model is on GPU but my labels might be on CPU."
}

user_prompt = f"""Classify this PyTorch debugging issue.

Question title:
{test_input["question_title"]}

Error text:
{test_input["error_text"]}

Code context:
{test_input["code_context"]}

Question body:
{test_input["question_body"]}

Return JSON only.
"""

prompt = (
    "<|system|>\n"
    f"{SYSTEM_PROMPT}<|end|>\n"
    "<|user|>\n"
    f"{user_prompt}<|end|>\n"
    "<|assistant|>\n"
)

model.eval()
model.config.use_cache = True

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("=== Fine-tuned Model Response ===")
print(response)

## Step 11 — Push Adapter to HuggingFace Hub

In [ ]:
HF_USERNAME = 'zehansunesara'  # your HF username
ADAPTER_REPO = 'pytorch-debug-assistant-phi3-structured-v3'

# Save and push the LoRA adapter (not the full model — adapter is only ~100MB)
if SMOKE_TEST:
    print("Smoke test only — skipping Hugging Face push.")
else:
    model.push_to_hub(f"{HF_USERNAME}/{ADAPTER_REPO}", token=HF_TOKEN)
    tokenizer.push_to_hub(f"{HF_USERNAME}/{ADAPTER_REPO}", token=HF_TOKEN)

    print(f"\n✅ Adapter pushed to huggingface.co/{HF_USERNAME}/{ADAPTER_REPO}")



## Step 12 — Save Training Loss Plot

In [ ]:
import matplotlib.pyplot as plt

# Extract loss history from trainer logs
logs = trainer.state.log_history
train_loss = [(l['step'], l['loss']) for l in logs if 'loss' in l]
eval_loss  = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(*zip(*train_loss), label='Train Loss', color='steelblue')
ax.plot(*zip(*eval_loss),  label='Val Loss',   color='coral', linestyle='--')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('QLoRA Fine-tuning: Phi-3-mini on PyTorch Debug Assistant')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print('Saved training_loss.png — add this to your README!')

## ✅ Phase 2 Complete!

**What you've done:**
- Fine-tuned Phi-3-mini with QLoRA on 809 high-quality PyTorch Q&A pairs
- Only ~1-2% of parameters were trained (LoRA adapters)
- Adapter pushed to HuggingFace Hub

**Next steps (Phase 3):**
- Apply GPTQ/AWQ post-training quantization
- Benchmark latency vs quality at different bit widths
- Build FastAPI + Gradio serving layer